
# PCA + LDA for Classification (MNIST + Digits)

Train/test split; fit reducers on **train only**; evaluate on **test**.


In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
np.random.seed(42)

def evaluate_pca_knn(X_train, X_test, y_train, y_test, dims, n_neighbors=3):
    rows = []
    for k in dims:
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("pca", PCA(n_components=k, random_state=42)),
                         ("knn", KNeighborsClassifier(n_neighbors=n_neighbors))])
        pipe.fit(X_train, y_train)
        rows.append({"method":"PCA+KNN","components":k,"accuracy":accuracy_score(y_test, pipe.predict(X_test))})
    return pd.DataFrame(rows)

def evaluate_lda_knn(X_train, X_test, y_train, y_test, dims, n_neighbors=3):
    rows = []
    for k in dims:
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("lda", LinearDiscriminantAnalysis(n_components=k, solver="svd")),
                         ("knn", KNeighborsClassifier(n_neighbors=n_neighbors))])
        pipe.fit(X_train, y_train)
        rows.append({"method":"LDA+KNN","components":k,"accuracy":accuracy_score(y_test, pipe.predict(X_test))})
    return pd.DataFrame(rows)

def lda_classifier_baseline(X_train, X_test, y_train, y_test):
    clf = LinearDiscriminantAnalysis(solver="svd")
    clf.fit(X_train, y_train)
    return accuracy_score(y_test, clf.predict(X_test))



## A) MNIST (requires internet on first run)


In [ ]:

from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(len(X_train), -1).astype("float32")/255.0
X_test  = X_test.reshape(len(X_test), -1).astype("float32")/255.0

pca_dims = [5, 10, 20, 50, 100, 200]
lda_dims = list(range(1, 10))  # up to 9

mnist_pca = evaluate_pca_knn(X_train, X_test, y_train, y_test, pca_dims, 3)
mnist_lda = evaluate_lda_knn(X_train, X_test, y_train, y_test, lda_dims, 3)
mnist_lda_clf = lda_classifier_baseline(X_train, X_test, y_train, y_test)

display(mnist_pca); display(mnist_lda); print("LDA classifier baseline:", mnist_lda_clf)

plt.figure(figsize=(7,5))
plt.plot(mnist_pca.sort_values("components")["components"], mnist_pca.sort_values("components")["accuracy"], marker="o", label="PCA+KNN")
plt.plot(mnist_lda.sort_values("components")["components"], mnist_lda.sort_values("components")["accuracy"], marker="o", label="LDA+KNN")
plt.xlabel("Number of Components"); plt.ylabel("Test Accuracy"); plt.title("MNIST: Accuracy vs Components"); plt.legend(); plt.tight_layout(); plt.show()



## B) sklearn Digits (offline-friendly)


In [ ]:

from sklearn.datasets import load_digits
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pca_dims = [5, 10, 20, 40, 60]
lda_dims = [1, 2, 3, 5, 7, 9]

digits_pca = evaluate_pca_knn(X_train, X_test, y_train, y_test, pca_dims, 3)
digits_lda = evaluate_lda_knn(X_train, X_test, y_train, y_test, lda_dims, 3)
digits_lda_clf = lda_classifier_baseline(X_train, X_test, y_train, y_test)

display(digits_pca); display(digits_lda); print("LDA classifier baseline:", digits_lda_clf)

plt.figure(figsize=(7,5))
plt.plot(digits_pca.sort_values("components")["components"], digits_pca.sort_values("components")["accuracy"], marker="o", label="PCA+KNN")
plt.plot(digits_lda.sort_values("components")["components"], digits_lda.sort_values("components")["accuracy"], marker="o", label="LDA+KNN")
plt.xlabel("Number of Components"); plt.ylabel("Test Accuracy"); plt.title("Digits: Accuracy vs Components"); plt.legend(); plt.tight_layout(); plt.show()
